# ForgeSavant model-readiness assessment

## tl;dr

In [1]:
from pathlib import Path
import json
import duckdb
import pandas as pd
from IPython.display import Markdown, display

project_dir = Path.cwd()
analytics_dir = project_dir / "data-pipeline" / "analytics"
readiness = json.loads((analytics_dir / "model_readiness_summary.json").read_text(encoding="utf-8"))
dataset = readiness["dataset"]
display(Markdown(
    f"**Descriptive quality monitoring is ready, but predictive ML is not.** "
    f"The warehouse contains **{dataset['distinctProducts']} distinct products** covering "
    f"**{dataset['catalogCoverageRate']:.1%}** of the {dataset['verifiedProducts']}-product catalog. "
    "It has no retailer-price series, performance targets, or user-outcome labels."
))

**Descriptive quality monitoring is ready, but predictive ML is not.** The warehouse contains **14 distinct products** covering **24.1%** of the 58-product catalog. It has no retailer-price series, performance targets, or user-outcome labels.

## Context & Methods

This diagnostic asks which analytical and machine-learning uses are supported by the current immutable product-content warehouse. It checks grain, key uniqueness, temporal validity, category coverage, label availability, and collection history.

### Key Assumptions

- Open Icecat observations are product-content evidence, not retailer offers or benchmark results.
- A known duplicate is a valid idempotent pipeline outcome and is not a second training row.
- Predictive readiness requires an independently observed target and an evaluation split appropriate to the intended use.

In [2]:
database_path = analytics_dir / "forgesavant.duckdb"
summary_path = analytics_dir / "data_quality_summary.json"
assert database_path.exists(), "Run npm run analytics:build first"
assert summary_path.exists(), "Run npm run analytics:build first"
connection = duckdb.connect(str(database_path), read_only=True)

## Data

The analytical grain is one immutable normalized source observation per source product content version.

In [3]:
profile = pd.DataFrame([{
    "Observations": dataset["observations"],
    "Unique observation IDs": dataset["distinctObservationIds"],
    "Distinct products": dataset["distinctProducts"],
    "Verified catalog": dataset["verifiedProducts"],
    "Collection dates": dataset["distinctIngestionDates"],
    "Future observations": dataset["futureObservations"],
    "Median specification fields": dataset["medianSpecificationFields"],
}])
display(profile)

assert dataset["observations"] == dataset["distinctObservationIds"]
assert dataset["futureObservations"] == 0

,Observations,Unique observation IDs,Distinct products,Verified catalog,Collection dates,Future observations,Median specification fields
0,28,28,14,58,1,0,46.0


## Results

### Coverage is concentrated in four categories

In [4]:
category_coverage = pd.DataFrame(readiness["categories"])
category_coverage["coverage"] = category_coverage["coverageRate"].map(lambda value: f"{value:.1%}" if value is not None else "n/a")
display(category_coverage[["category", "observedProducts", "verifiedProducts", "coverage"]])

,category,observedProducts,verifiedProducts,coverage
0,cabinets,0,2,0.0%
1,gpus,6,11,54.5%
2,motherboards,3,12,25.0%
3,power_supplies,0,6,0.0%
4,processors,0,13,0.0%
5,ram,3,10,30.0%
6,storage,2,4,50.0%


### Current evidence supports monitoring, not prediction

In [5]:
use_readiness = pd.DataFrame(readiness["uses"])
display(use_readiness)
check_results = pd.DataFrame([{"check": key, "passed": value} for key, value in readiness["checks"].items()])
display(check_results)

,use,status,reason
0,Descriptive data-quality monitoring,ready,"Validated manifests, immutable observations, a..."
1,Product-content enrichment pilot,limited,Only 14 of 58 verified products have accepted ...
2,Supervised build recommendation model,blocked,"There are no observed user outcomes, benchmark..."
3,India price prediction or forecasting,blocked,"Open Icecat supplies product content, not reta..."


,check,passed
0,uniqueObservationIds,True
1,noFutureObservations,True
2,allCategoriesObserved,False
3,supervisedOutcomeLabelsPresent,False
4,temporalHistoryAtLeastEightDates,False


## Takeaways

- Use the current warehouse for provenance, completeness, source-coverage, and ingestion-health analysis.
- Pilot reviewed product-content enrichment only on the 14 covered products; do not infer missing categories from them.
- Keep the existing rule-based compatibility engine as the production decision layer until labeled outcomes exist.
- Do not train or advertise a price forecast, performance predictor, or personalized recommender from this dataset.

In [6]:
display(Markdown("### Evidence required next\n" + "\n".join(f"- {item}" for item in readiness["requiredNextEvidence"])))
connection.close()

### Evidence required next
- Authorized retailer offer snapshots with product identity, INR price, availability, retailer, and observation time.
- Independent performance benchmark observations at a declared workload, resolution, settings, and test date.
- Consented product interaction and saved-build outcomes before any personalized recommendation model.
- Repeated collection dates and leakage-safe train/validation/test splits before temporal evaluation.